# Effective Inference — Kaggle 2×T4 пайплайн (SFT + self-distill)

Цель: дообучить маленькую модель (Qwen3-**4B** приоритет, **1.7B** запасной) отвечать на школьные
вопросы так же хорошо, как большой учитель, и уложиться в контестные лимиты на **L4** (4000 вопросов < 15 мин).

**Что делает этот ноутбук (всё на 2×T4 16GB, Turing → fp16, без bf16/FlashAttn2):**
1. **Подготовка данных** — train/held-out сплит из `dataset_ml_challenge.parquet`.
2. **SFT** базовой модели на эталонах — **DDP на 2 карты** (QLoRA nf4, mask промпта, NEFTune).
3. **Self-distill (RAFT)** — SFT-моделью генерим N кандидатов на train (**data-parallel на 2 карты**),
   отбираем лучшие (эвристики-фильтр + локальный судья Qwen2.5-3B на 2 карты), дообучаем на наборе «выше эталона».
4. **AWQ-квантизация** смерженной модели → веса под L4.
5. **Оффлайн-оценка** — win-rate против эталона + экстраполяция тайминга (не жечь 4 сабмита/сутки на догадках).
6. **Упаковка** весов в `weights/` для посылки.

> Тяжёлые GPU-шаги запускаются как `!python` / `!accelerate launch` / `subprocess` (subprocess), чтобы vLLM/трейнер
> освобождали видеопамять между этапами — в одном ядре ноутбука это иначе не выгрузить.

**Использование 2×T4:** SFT — DDP (по копии модели на карту, ×2 throughput). Генерация и судья —
data-parallel (два процесса по карте, шардинг запросов): на T4 без NVLink это быстрее tensor-parallel.

**Перед запуском:** Settings → Accelerator = **GPU T4 ×2**, Internet = **ON**. Добавь как Kaggle Dataset
папку `EffectiveInference` (с `dataset_ml_challenge.parquet` и `effinf_dev/`) — путь укажи в ячейке Config.

## 0. Установка зависимостей

In [ ]:
# vLLM тянет совместимые torch/transformers. autoawq — для квантизации под L4.
# bitsandbytes>=0.46.1 — иначе transformers ругается на 4-bit QLoRA.
!pip install -q -U vllm==0.6.3.post1 transformers==4.46.* peft==0.13.* trl==0.11.* \
    accelerate==1.0.* "bitsandbytes>=0.46.1" autoawq==0.2.6 datasets pyarrow
print('deps installed — при конфликтах версий перезапусти ядро и продолжай со следующей ячейки')

## 1. Config, рабочий каталог и helper для data-parallel генерации

In [ ]:
import os, shutil, subprocess, glob

# Автопоиск папки EffectiveInference в подключённых датасетах (слаг ≠ отображаемое имя).
def find_src():
    # ищем каталог, где есть и effinf_dev/, и dataset_ml_challenge.parquet
    for parquet in glob.glob('/kaggle/input/**/dataset_ml_challenge.parquet', recursive=True):
        d = os.path.dirname(parquet)
        if os.path.isdir(os.path.join(d, 'effinf_dev')):
            return d
    # fallback: любая папка effinf_dev
    for dev in glob.glob('/kaggle/input/**/effinf_dev', recursive=True):
        return os.path.dirname(dev)
    raise FileNotFoundError('Не нашёл EffectiveInference в /kaggle/input — проверь Add Input')

SRC = find_src()
WORK = '/kaggle/working/EffectiveInference'
print('SRC =', SRC)

# Копируем repo в writable working (input — read-only).
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK)
DEV = os.path.join(WORK, 'effinf_dev')
os.chdir(DEV)
print('cwd =', os.getcwd())

# --- гиперпараметры пайплайна ---
BASE_4B   = 'Qwen/Qwen3-4B'
BASE_1_7B = 'Qwen/Qwen3-1.7B'
EPOCHS    = 2
MAX_LEN   = 2048
DISTILL_LIMIT = 4000   # сколько train-запросов прогнать через self-distill
DISTILL_N     = 6      # кандидатов на запрос
JUDGE_MODEL   = 'Qwen/Qwen2.5-3B-Instruct'

os.makedirs('work', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def gen_distill_dp(model_dir, out, n, limit, max_tokens=1280, num_gpus=2):
    """Data-parallel генерация: num_gpus процессов, по одной карте на каждый
    (CUDA_VISIBLE_DEVICES), запросы шардятся, шарды склеиваются. Быстрее TP=2 на
    T4 без NVLink (нет межкарточного обмена). Каждый процесс на выходе освобождает VRAM."""
    shards, procs = [], []
    for g in range(num_gpus):
        sh = f'{out}.shard{g}'
        shards.append(sh)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        cmd = (f'python gen_distill.py --model_dir {model_dir} --train_jsonl data/train_minimal.jsonl '
               f'--out {sh} --n {n} --limit {limit} --max_tokens {max_tokens} '
               f'--tensor_parallel_size 1 --num_shards {num_gpus} --shard_id {g}')
        procs.append(subprocess.Popen(cmd, shell=True, env=env))
    for p in procs:
        assert p.wait() == 0, 'shard упал — см. лог выше'
    with open(out, 'w', encoding='utf-8') as fo:
        for sh in shards:
            with open(sh, encoding='utf-8') as fi:
                fo.write(fi.read())
    print('склеено →', out)

In [ ]:
# prepare_data.py читает ../dataset_ml_challenge.parquet (на уровень выше effinf_dev) и
# пишет data/{train_minimal,train_rich,eval}.jsonl + splits.json (детерминир. held-out 800).
!python prepare_data.py
!wc -l data/train_minimal.jsonl data/eval.jsonl

## 2. SFT 4B (QLoRA nf4 + DDP на 2×T4)

Loss только по токенам ответа (промпт замаскирован). DDP: по копии модели на карту, ~×2 throughput.
~9.2k примеров × 2 эпохи. Если сессия близка к лимиту — `--resume` докатывает с чекпоинта.

> Если DDP закапризничает на Kaggle — замени `accelerate launch ...` на `!python kaggle_sft.py ...` (1 карта, медленнее, но надёжно).

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 \
    kaggle_sft.py \
    --train_jsonl data/train_minimal.jsonl \
    --output_dir work/lora_4b \
    --base_model $BASE_4B \
    --epochs 2 --max_len 2048 --neftune 5.0

In [ ]:
# Мерж адаптера в полные fp16-веса (vLLM грузит как обычную модель).
!python merge_lora.py --base_model $BASE_4B --adapter work/lora_4b --out work/merged_4b

## 3. Self-distill (RAFT) для 4B

**3a.** SFT-моделью генерим `DISTILL_N` кандидатов на каждый из `DISTILL_LIMIT` train-запросов (temperature sampling, **data-parallel на 2 карты**).

**3b.** Отбор: эвристики выкидывают обрезанные/битые/зацикленные → локальный судья (TP=2) ранжирует выживших против эталона → собираем набор «не ниже эталона».

**3c.** Дообучаем с нуля от базы на RAFT-наборе (чище, чем продолжать с того же адаптера).

In [ ]:
# data-parallel: два процесса по карте, шардинг запросов, склейка (быстрее TP=2 на T4 без NVLink).
gen_distill_dp('work/merged_4b', 'data/distill_cand_4b.jsonl', DISTILL_N, DISTILL_LIMIT, max_tokens=1280)

In [ ]:
# Судья на ОБЕИХ картах (TP=2). Печатает % замен эталона — сигнал пользы distill.
!python select_distill.py \
    --candidates data/distill_cand_4b.jsonl \
    --out data/raft_4b.jsonl \
    --judge_model $JUDGE_MODEL --topk 2 --tensor_parallel_size 2

In [ ]:
# RAFT-набор покрывает только DISTILL_LIMIT запросов. Дообучаем на нём поверх остального train:
# объединяем raft (заменённые/подтверждённые) с хвостом train_minimal, которого distill не касался.
import json
raft = [json.loads(l) for l in open('data/raft_4b.jsonl', encoding='utf-8')]
tail = [json.loads(l) for l in open('data/train_minimal.jsonl', encoding='utf-8')][DISTILL_LIMIT:]
with open('data/sft2_4b.jsonl', 'w', encoding='utf-8') as f:
    for r in raft + tail:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print('SFT-2 набор:', len(raft) + len(tail), 'примеров')

In [ ]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 \
    kaggle_sft.py \
    --train_jsonl data/sft2_4b.jsonl \
    --output_dir work/lora_4b_raft \
    --base_model $BASE_4B \
    --epochs 2 --max_len 2048 --neftune 5.0

!python merge_lora.py --base_model $BASE_4B --adapter work/lora_4b_raft --out work/merged_4b_raft

## 4. AWQ-квантизация 4B (веса под L4)

Калибровка на реальных train-примерах в chat-формате, чтобы int4 не «уплыл» на школьном домене.
AWQ грузится и на T4 (для оффлайн-проверки ниже), и на L4 в посылке.

In [ ]:
!python quantize_awq.py \
    --model work/merged_4b_raft \
    --out work/awq_4b \
    --calib_jsonl data/train_minimal.jsonl --calib_n 256

## 5. Оффлайн-оценка 4B (win-rate + тайминг)

Генерим ответы на held-out (800), считаем win-rate против эталона локальным судьёй и смотрим
экстраполяцию времени на 4000 запросов. NB: тайминг на T4 ≠ L4 (L4 быстрее), но даёт верхнюю оценку.

In [ ]:
# gen_candidates печатает tok/s, среднюю длину, % упёршихся в max_tokens и экстраполяцию на 4000.
!python gen_candidates.py \
    --model_dir work/awq_4b --variant minimal \
    --eval_jsonl data/eval.jsonl --out data/cand_awq_4b.jsonl \
    --max_tokens 1280 --dtype float16

In [ ]:
!python judge_local.py --candidates data/cand_awq_4b.jsonl \
    --judge_model $JUDGE_MODEL --dtype float16

## 6. (Запасной) 1.7B — тот же конвейер, TL-безопасный

Меньше, быстрее на L4 (страховка от TimeLimit), качество ниже. Гоняем те же шаги и сравниваем win-rate
с 4B оффлайн-судьёй. Поставь `RUN_1_7B = True` и запусти, если хочешь иметь обе посылки.

In [ ]:
RUN_1_7B = False
if RUN_1_7B:
    subprocess.run('accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 '
        'kaggle_sft.py --train_jsonl data/train_minimal.jsonl --output_dir work/lora_1_7b '
        f'--base_model {BASE_1_7B} --epochs 2 --max_len 2048 --neftune 5.0', shell=True, check=True)
    subprocess.run(f'python merge_lora.py --base_model {BASE_1_7B} --adapter work/lora_1_7b --out work/merged_1_7b', shell=True, check=True)
    # data-parallel генерация на 2 карты
    gen_distill_dp('work/merged_1_7b', 'data/distill_cand_1_7b.jsonl', DISTILL_N, DISTILL_LIMIT, max_tokens=1280)
    subprocess.run(f'python select_distill.py --candidates data/distill_cand_1_7b.jsonl --out data/raft_1_7b.jsonl --judge_model {JUDGE_MODEL} --topk 2 --tensor_parallel_size 2', shell=True, check=True)
    raft = [json.loads(l) for l in open('data/raft_1_7b.jsonl', encoding='utf-8')]
    tail = [json.loads(l) for l in open('data/train_minimal.jsonl', encoding='utf-8')][DISTILL_LIMIT:]
    with open('data/sft2_1_7b.jsonl','w',encoding='utf-8') as f:
        for r in raft+tail: f.write(json.dumps(r, ensure_ascii=False)+'\n')
    subprocess.run('accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 '
        'kaggle_sft.py --train_jsonl data/sft2_1_7b.jsonl --output_dir work/lora_1_7b_raft '
        f'--base_model {BASE_1_7B} --epochs 2 --max_len 2048 --neftune 5.0', shell=True, check=True)
    subprocess.run(f'python merge_lora.py --base_model {BASE_1_7B} --adapter work/lora_1_7b_raft --out work/merged_1_7b_raft', shell=True, check=True)
    subprocess.run('python quantize_awq.py --model work/merged_1_7b_raft --out work/awq_1_7b --calib_jsonl data/train_minimal.jsonl --calib_n 256', shell=True, check=True)
    subprocess.run('python gen_candidates.py --model_dir work/awq_1_7b --variant minimal --eval_jsonl data/eval.jsonl --out data/cand_awq_1_7b.jsonl --max_tokens 1280 --dtype float16', shell=True, check=True)
    subprocess.run(f'python judge_local.py --candidates data/cand_awq_1_7b.jsonl --judge_model {JUDGE_MODEL} --dtype float16', shell=True, check=True)

## 7. Упаковка весов для посылки

Кладём выбранную AWQ-модель в `weights/` посылки (заменяя стоковые веса). Сам код посылки
(`solution.py`, `source/`, `Dockerfile`) у тебя уже готов — здесь только веса.
Скачай `work/awq_4b/` из Output ноутбука и положи в `EffectiveInference/weights/` перед сборкой zip.

In [ ]:
BEST = 'work/awq_4b'   # поменяй на work/awq_1_7b, если запасная победила по win-rate/таймингу
!ls -la $BEST
!du -sh $BEST
print('\nГотово. Скачай папку из Output и положи её содержимое в weights/ посылки.')